In [1]:
import os
import pwd
import numpy as np
import pandas as pd
import sys
from pyspark.sql import Window


from pyspark.sql import SparkSession
from random import randrange
import pyspark.sql.functions as F


username = pwd.getpwuid(os.getuid()).pw_name
hadoopFS=os.getenv('HADOOP_FS', None)
groupName = "H1"

print(os.getenv('SPARK_HOME'))
print(f"hadoopFSs={hadoopFS}")
print(f"username={username}")
print(f"group={groupName}")

/opt/spark
hadoopFSs=hdfs://iccluster061.iccluster.epfl.ch:9000
username=hsari
group=H1


In [2]:
spark = (SparkSession\
            .builder
            .appName(username + '-assignment-2')
            .config('spark.ui.port', randrange(4050, 4450, 5))
            .config("spark.executorEnv.PYTHONPATH", ":".join(sys.path))
            .config('spark.jars',
                    f'{hadoopFS}/data/com-490/jars/iceberg-spark-runtime-3.5_2.13-1.6.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/sedona-spark-shaded-3.5_2.13-1.7.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/geotools-wrapper-1.7.1-28.5.jar'
            )
            .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
            .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
            .config('spark.sql.catalog.iceberg.type', 'hadoop')
            .config('spark.sql.catalog.iceberg.warehouse', f'{hadoopFS}/data/com-490/silver/')
            .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
            .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
            .config('spark.sql.catalog.spark_catalog.warehouse', f'{hadoopFS}/user/{username}/assignment-3/warehouse')
            .config("spark.sql.warehouse.dir", f'{hadoopFS}/user/{username}/assignment-3/spark/warehouse')
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.executor.instances", "4")
        ).master('yarn').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 15:43:08 WARN Utils: Service 'SparkUI' could not bind on port 4085. Attempting port 4086.
26/05/16 15:43:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
base = f"{hadoopFS}/user/groups/com-490/H1/final/v1"

## From delay_oracle.py

In [5]:
from delay_lookup import init_delay_oracle, delay_prob_from_timetable_row
base = f"{hadoopFS}/user/groups/com-490/H1/final/v1"
LOOKUP_PATH = f"{base}/route_demo_lookup_february.parquet"

init_delay_oracle(spark, LOOKUP_PATH)
FEB_TIMETABLE_PATH = f"{base}/timetable_february.parquet"

feb_raw_df = spark.read.parquet(FEB_TIMETABLE_PATH)

In [6]:
row = (
    feb_raw_df
    .filter(F.col("arrival_timestamp").isNotNull())
    .limit(1)
    .first()
)

p = delay_prob_from_timetable_row(row, X=180)

print("P(delay <= 180 sec):", p)

P(delay <= 180 sec): 0.8300431625406198


In [7]:
spark.stop()